[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tariqchoucair/newsvoice/blob/main/demo.ipynb)

# newsvoice - example

Voice and attribution extraction from news text.
This notebook demonstrates the newsvoice package on a synthetic article.

## Setup

Our pipeline was developed using the spacy transformer model `en_core_web_trf`. Swap in
`en_core_web_sm` for a fast run, but note that attribution accuracy degrades on smaller models.


In [ ]:
%pip install -q git+https://github.com/tariqchoucair/newsvoice.git
%pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_trf-3.8.0/en_core_web_trf-3.8.0-py3-none-any.whl

In [ ]:
import json

import pandas as pd

import newsvoice

MODEL = "en_core_web_trf"   # or "en_core_web_sm" for a fast run

nlp = newsvoice.load_pipeline(MODEL)
print(f"newsvoice {newsvoice.__version__}  model={MODEL}")
print(newsvoice.DEFAULT_CONFIG)

## Example

The article exercises multiple constructions the pipeline is built for.

In [ ]:
ARTICLE = """Energy regulator warns of price rises

The Australian Energy Regulator said household bills would climb by nine per
cent, describing the increase as "unavoidable" in the current market.

"Families are already stretched to breaking point," said Ms Priya Raman, the
regulator's chief executive.

"We have looked at every alternative over the past eleven months.

"There is no version of this decision that does not hurt someone."

The AER declined to say whether the increase would be reviewed.

Dr Alan Whitfield, of Northfield University, warned of further rises. The
economist added that the modelling had understated network costs for a decade.

The department did not reveal the forecasts behind the decision.
"""

print(ARTICLE)


In [ ]:
rows = newsvoice.extract_document("demo-1", ARTICLE, nlp)
print(f"{len(rows)} attributions\n")

for row in rows:
    print(f"{row['Actor Canonical Name']}  ({row['Actor Entity Type']})\n"
          f"  mention:      {row['Speaker Mention']}\n"
          f"  cues:         {', '.join(json.loads(row['Attribution Cues (JSON)']))}\n"
          f"  voice type:   {row['Voice Type']}\n"
          f"  explicitness: {row['Attribution Explicitness']}\n"
          f"  position:     {row['Attribution Position']}")
    for segment in json.loads(row["Direct Segments (JSON)"]):
        print(f"    direct   | {segment}")
    for segment in json.loads(row["Indirect Segments (JSON)"]):
        print(f"    indirect | {segment}")
    print()


### Things to check in that output

- The headline carries `warns`, but produces no row.
- `did not reveal` produces no row — a non-disclosure gives nobody voice.
- `the regulator's chief executive` and `The economist` resolve to named people.
- `The AER` expands to the full organisation name.
- The three consecutive quotation paragraphs stay with Ms Raman, even though
  only the first carries an explicit cue.
- `Speaker Mention` keeps the surface form; `Actor Canonical Name` records the
  inference. They are separate columns so the inference stays auditable.


## Traceability

Every row carries offsets into the exact string passed in. This is the property
the three-column design exists to preserve: no analytical object is detached
from the discourse it came from.


In [ ]:
for row in rows[:3]:
    start = row["Evidence Start Character (0-based)"]
    end = row["Evidence End Character (exclusive)"]
    assert ARTICLE[start:end] == row["Evidence Span"]
    print(f"[{start:>4}:{end:<4}] {row['Evidence Span'][:90]}...")

print("\nall offsets verified against the source")


## Inspecting one layer

Each layer is importable on its own, which is how you diagnose a row that looks
wrong: work upwards until you find the layer that first gets it wrong.


In [ ]:
from newsvoice.cues import find_cues
from newsvoice.quotes import find_quote_spans
from newsvoice.syntax import cue_subject

sentence = '"Costs will rise," said Ms Priya Raman, the chief executive.'
doc = nlp(sentence)

print("quotes: ", [sentence[a:b] for a, b in find_quote_spans(sentence)])
for span, text, tier in find_cues(doc):
    speaker = cue_subject(span, doc)
    print(f"cue:     {text!r} ({tier})  speaker={speaker!r}")


## Configuration

The tunables are researcher degrees of freedom, not implementation details.
`orphan_recovery` in particular sets an attribution standard: on, it follows the
news convention that a reader carries attribution across a paragraph break; off,
it requires explicit attribution and yields fewer, more defensible rows.

Record `config.as_dict()` with your results.


In [ ]:
from newsvoice import ExtractionConfig

strict = ExtractionConfig(orphan_recovery=False)
strict_rows = newsvoice.extract_document("demo-1", ARTICLE, nlp, config=strict)

print(f"default:  {len(rows)} rows")
print(f"strict:   {len(strict_rows)} rows")
print(f"\nconfig recorded as: {json.dumps(strict.as_dict())}")


## Your corpus

`extract_corpus` takes a DataFrame with an id column and a text column. Nothing
else is read. Documents that raise are recorded with
`Processing Status = "error"` rather than lost; pass `on_error="raise"` to stop
on the first failure while debugging a new corpus.


In [ ]:
# articles = pd.read_csv("articles.csv")

articles = pd.DataFrame({
    "article_id": ["demo-1", "demo-2"],
    "full_text": [ARTICLE, "Prices rose. Nobody was quoted."],
})

quotes = newsvoice.extract_corpus(
    articles,
    nlp,
    id_column="article_id",
    text_column="full_text",
    progress=True,
)

print(f"{len(quotes):,} rows from {len(articles):,} articles")
quotes[["Article Id", "Actor Canonical Name", "Actor Entity Type",
        "Voice Type", "Attribution Explicitness", "Processing Status"]]


In [ ]:
quotes.to_csv("quotes.csv", index=False)
print("wrote quotes.csv")


## Before you publish anything from this

The test suite pins behaviour on synthetic text. It does not establish accuracy
on real news, and the two are different claims.

Hand-code a random sample of *your own* corpus and report agreement against it.
Draw it from the corpus actually analysed, not from a benchmark or a convenience
set of clear cases. Attribution error concentrates in passives, in epithets and
in multi-paragraph quotation turns, so a sample stratified over those
constructions is worth more than a random one of the same size.

See `docs/KNOWN_ISSUES.md` for constructions that are wrong in known ways.
